# Analisis de situaciones
investigar cada situacion con datos concretos y elegir la decision mas acertada para el pipeline Silver

Regla general: Silver **documenta y enriquece**; no reescribe el origen sin evidencia clara.

## Configuracion

In [1]:
import json
import re
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP

import pandas as pd

pd.set_option("display.max_rows", None)

PROJECT_ROOT = Path("/home/jovyan/work")
RAW_PATH = PROJECT_ROOT / "data" / "raw"

with open(PROJECT_ROOT / "manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)


def load_raw(domain, table):
    path = RAW_PATH / domain / f"{table}.csv"
    return pd.read_csv(path)


def to_decimal(x):
    return Decimal(str(x))


def to_money(x):
    return to_decimal(x).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/jovyan/work


## Situaciones a resolver

| # | Situacion |
|---|-----------|
| 1 | Emails duplicados en `contacts.email` |
| 2 | Fechas incoherentes en `subscriptions` (`end_date < start_date`) |
| 3 | `invoice_items.line_total` != `quantity * unit_price` |
| 4 | `invoices.total` != suma de `invoice_items` |
| 5 | Facturas sin `invoice_items` |
| 6 | Facturas sobrepagadas |
| 7 | Facturas `status=paid` con pago < total |
| 8 | Suma de `grades.weight` por enrollment != 1 |

## 1. Emails duplicados en contacts

In [12]:
contacts = load_raw("crm", "contacts")
dup = contacts[contacts["email"].duplicated(keep=False)].sort_values("email")

print("Filas con email repetido:", len(dup))
print("Emails distintos repetidos:", dup["email"].nunique())
print("Cuentas distintas involucradas:", dup["account_id"].nunique())
dup[["contact_id", "email", "account_id"]]

Filas con email repetido: 4
Emails distintos repetidos: 2
Cuentas distintas involucradas: 4


,contact_id,email,account_id
2466,CON-0002467,ignacio.tapia5946@mail.test,ACC-0002715
8391,CON-0008392,ignacio.tapia5946@mail.test,ACC-0004626
3103,CON-0003104,macarena.ortiz4996@lake.local,ACC-0003066
4934,CON-0004935,macarena.ortiz4996@lake.local,ACC-0001325


Los 2 emails repetidos en 4 contactos de cuentas distintas, no hay PK duplicada ni fila identica.  
**Decision** : conservar las filas sin modificacion ni columnas adicionales. Son contactos distintos de cuentas distintas; el email repetido puede ser dato real (email compartido) o error de origen.

## 2. Fechas incoherentes en subscriptions

In [4]:
subscriptions = load_raw("billing", "subscriptions")
subs = subscriptions.copy()
subs["start_dt"] = pd.to_datetime(subs["start_date"])
subs["end_dt"] = pd.to_datetime(subs["end_date"])
bad_subs = subs[subs["end_dt"] < subs["start_dt"]]

print("Filas con end_date < start_date:", len(bad_subs))
print("\nPor status:")
print(bad_subs["status"].value_counts())


Filas con end_date < start_date: 783

Por status:
status
active       583
cancelled    103
paused        97
Name: count, dtype: int64


783 suscripciones tienen el rango invertido. Al intercambiar `start_date` y `end_date`, todas pasan a ser coherentes

**Decision:** Intercambiar `start_date` y `end_date` directamente en las filas con rango invertido. Sin columnas adicionales: Silver entrega fechas ya coherentes.

### 2.1 Evidencia adicional: ¿el swap es una decision razonable?

La decision actual es *marcar y no corregir* porque falta una regla de negocio escrita. Para tener mas criterio, revisamos dos señales simples sobre las 783 filas invertidas:

1. ¿En que rango de fechas ocurren?
        Si se concentran en un periodo puntual (en vez de repartirse en todo el historico), es mas probable que sea un problema sistematico y no ruido aleatorio.
3. ¿La cantidad de casos aumenta con el tiempo?
        Si el problema crece anio a anio, refuerza la idea de que empezo en cierto momento y se mantuvo, en vez de ser un error disperso al azar.

In [6]:
anios = range(subs["start_dt"].dt.year.min(), subs["start_dt"].dt.year.max() + 1)
casos_por_anio = bad_subs["start_dt"].dt.year.value_counts().reindex(anios, fill_value=0).sort_index()

print("Casos de fechas invertidas por año de start_date:")
print(casos_por_anio)

Casos de fechas invertidas por año de start_date:
start_dt
2020      0
2021      0
2022      0
2023      0
2024    350
2025    433
Name: count, dtype: int64


**DESCUBRIMIENTO**  
Los casos invertidos solo aparecen en 2024 y 2025, 0 casos en 2020-2023, y ademas crecen de un año a otro (350 en 2024, 433 en 2025). Estas dos señales juntas — un rango de fechas acotado y una tendencia creciente — apuntan a un problema sistematico que aparecio en cierto momento.


**Decision (refinada):** Intercambiar `start_date` y `end_date` in place en las 783 filas afectadas. La evidencia de error sistematico (2024-2025, tendencia creciente) respalda la correccion automatica; Silver no conserva el valor original ni agrega columnas de auditoria.

## 3. line_total vs quantity * unit_price

In [11]:
invoice_items = load_raw("billing", "invoice_items")
ii = invoice_items.copy()
ii["calc_dec"] = ii["quantity"].apply(to_decimal) * ii["unit_price"].apply(to_decimal)
ii["line_dec"] = ii["line_total"].apply(to_decimal)

ii["mismatch_strict"] = ii["line_dec"] != ii["calc_dec"]
ii["mismatch_money"] = ii["line_total"].apply(to_money) != ii["calc_dec"].apply(to_money)

print("Desajuste comparando Decimal directo:", ii["mismatch_strict"].sum())
print("Desajuste comparando montos a 2 decimales:", ii["mismatch_money"].sum())

m = ii[ii["mismatch_strict"]].head()
for _, row in m.iterrows():
    q = to_decimal(row["quantity"])
    p = to_decimal(row["unit_price"])
    print(f"quantity={q} unit_price={p} calc_total={q*p} line_total={row['line_total']}")

Desajuste comparando Decimal directo: 10668
Desajuste comparando montos a 2 decimales: 0
quantity=7 unit_price=64.65 calc_total=452.55 line_total=452.55000000000007
quantity=9 unit_price=59.88 calc_total=538.92 line_total=538.9200000000001
quantity=7 unit_price=25.49 calc_total=178.43 line_total=178.42999999999998
quantity=7 unit_price=25.35 calc_total=177.45 line_total=177.45000000000002
quantity=6 unit_price=32.88 calc_total=197.28 line_total=197.28000000000003


Con *Decimal* estricto hay 10 668 filas distintas, pero 0 al redondear a centavos. Es ruido de punto flotante del CSV, no error de negocio.

**Decision:** Reemplazar `line_total` por `quantity * unit_price` calculado con `Decimal` y redondeo a 2 decimales. Es ruido de punto flotante del CSV.

## 4. invoices.total vs suma de items

In [14]:
invoices = load_raw("billing", "invoices")

items_sum = (
    invoice_items.groupby("invoice_id")["line_total"]
    .apply(lambda s: sum(to_money(x) for x in s))
    .reset_index(name="items_sum")
)

inv_recon = invoices.merge(items_sum, on="invoice_id", how="left")
inv_recon["items_sum"] = inv_recon["items_sum"].apply(lambda x: Decimal("0") if pd.isna(x) else x)
inv_recon["total_money"] = inv_recon["total"].apply(to_money)
inv_recon["diff"] = inv_recon["total_money"] - inv_recon["items_sum"]

match = (inv_recon["total_money"] == inv_recon["items_sum"]).sum()
print(f"Facturas con total = suma items (centavos): {match}/{len(inv_recon)}")

inv_recon["diff_float"] = inv_recon["diff"].apply(float)
inv_recon.nlargest(5, "diff_float")[["invoice_id", "total", "items_sum", "diff", "status"]]

Facturas con total = suma items (centavos): 1/50000


,invoice_id,total,items_sum,diff,status
42680,INV-00042681,4348.24,576.09,3772.15,pending
9107,INV-00009108,3157.51,135.90,3021.61,paid
36542,INV-00036543,3994.28,1366.25,2628.03,paid
7301,INV-00007302,2733.95,378.69,2355.26,paid
47791,INV-00047792,2939.01,607.81,2331.20,pending


Solo 1 factura cuadra exacto, no es precision flotante el campo `total` del origen no refleja la suma de lineas (posible impuesto, descuento global o error de sistema).

**Decision:** 
Reemplazar `total` por la suma Decimal de sus `invoice_items` (redondeada a centavos) cuando no cuadre. Silver entrega facturas cuyo total refleja la suma real de lineas, sin columnas de reconciliacion.

## 5. Facturas sin invoice_items

In [17]:
ids_con_items = set(invoice_items["invoice_id"])
sin_items = invoices[~invoices["invoice_id"].isin(ids_con_items)]

print("Facturas sin items:", len(sin_items))

sin_items["status"].value_counts().to_frame()

Facturas sin items: 2502


,count
status,
paid,1774
pending,481
overdue,247


2 502 facturas no tienen lineas pero todas tienen `total > 0` y muchas categorizadas como `paid`. Probable factura cabecera sin detalle exportado o carga incompleta.

**Decision:** Excluir del output Silver las 2 502 facturas sin `invoice_items`. Sin detalle no son analiticamente utilizables; la tabla resultante solo incluye facturas con al menos una linea.

### 5.1 Evidencia adicional: ¿el problema crece con el tiempo?

Revisamos si la proporcion de facturas sin items aumenta en los periodos mas recientes (lo que sugeriria una falla reciente del pipeline de carga) o se mantiene estable en todo el historico.

In [18]:
inv_all = invoices.copy()
inv_all["sin_items"] = ~inv_all["invoice_id"].isin(ids_con_items)
inv_all["issued_year"] = pd.to_datetime(inv_all["issued_at"]).dt.year

print("Tasa de facturas sin items por anio de issued_at:")
print(inv_all.groupby("issued_year")["sin_items"].mean())

Tasa de facturas sin items por anio de issued_at:
issued_year
2022    0.049212
2023    0.050336
2024    0.048378
2025    0.052247
Name: sin_items, dtype: float64


La tasa se mantiene estable entre 4.8% y 5.2% en los 4 años (2022-2025), sin tendencia creciente ni decreciente. No es una falla reciente del pipeline: es una caracteristica constante del origen de datos.

**Decision (confirmada):** Excluir del output Silver las facturas sin items. La tasa estable (~5%) confirma que es caracteristica del origen, no una falla reciente; la limpieza consiste en omitir esas filas para entregar una tabla consistente.

## 6. Facturas sobrepagadas

In [23]:
payments = load_raw("billing", "payments")

pay_sum = (
    payments.groupby("invoice_id")["amount"]
    .apply(lambda s: sum(to_money(x) for x in s))
    .reset_index(name="paid_sum")
)

pay_recon = invoices.merge(pay_sum, on="invoice_id", how="left")
pay_recon["paid_sum"] = pay_recon["paid_sum"].apply(lambda x: Decimal("0") if pd.isna(x) else x)
pay_recon["total_money"] = pay_recon["total"].apply(to_money)

overpaid = pay_recon[pay_recon["paid_sum"] > pay_recon["total_money"]]
print("Sobrepagadas:", len(overpaid), f"({100*len(overpaid)/len(pay_recon):.1f}%)")
print("Pagos por factura (sobrepagadas):")
overpaid.head(5)[["invoice_id", "status", "total", "paid_sum"]]

Sobrepagadas: 20483 (41.0%)
Pagos por factura (sobrepagadas):


,invoice_id,status,total,paid_sum
1,INV-00000002,paid,70.06,166.14
2,INV-00000003,paid,35.12,36.32
3,INV-00000004,paid,16.19,41.49
4,INV-00000005,paid,27.62,52.02
8,INV-00000009,paid,47.66,70.93


20483 facturas recibieron mas pago que el total. Muchas tienen multiples pagos (pagos parciales acumulados, ajustes o datos inconsistentes con el total).

**Decision:** Ajustar los montos en `payments` para que la suma por `invoice_id` no supere el `total` de la factura, editando los valores.

### 6.1 Evidencia adicional: ¿que explica el sobrepago?

Ademas de revisar si el problema crece con el tiempo, probamos una hipotesis simple: ¿el sobrepago esta relacionado con la cantidad de pagos que recibio la factura?

In [21]:
pay_count = payments.groupby("invoice_id").size().rename("n_pagos")
pay_recon2 = pay_recon.merge(pay_count, on="invoice_id", how="left")
pay_recon2["n_pagos"] = pay_recon2["n_pagos"].fillna(0)
pay_recon2["issued_year"] = pd.to_datetime(pay_recon2["issued_at"]).dt.year
pay_recon2["overpaid"] = pay_recon2["paid_sum"] > pay_recon2["total_money"]

print("Tasa de sobrepago por anio de issued_at:")
print(pay_recon2.groupby("issued_year")["overpaid"].mean())

print("\nTasa de sobrepago segun cantidad de pagos recibidos:")
print(pay_recon2.groupby("n_pagos")["overpaid"].mean())

Tasa de sobrepago por anio de issued_at:
issued_year
2022    0.412499
2023    0.408211
2024    0.409261
2025    0.408668
Name: overpaid, dtype: float64

Tasa de sobrepago segun cantidad de pagos recibidos:
n_pagos
0.0     0.000000
1.0     0.000000
2.0     0.716272
3.0     0.983465
4.0     0.999502
5.0     1.000000
6.0     1.000000
7.0     1.000000
8.0     1.000000
9.0     1.000000
10.0    1.000000
11.0    1.000000
Name: overpaid, dtype: float64


La tasa de sobrepago es estable en el tiempo con una media de 0.41, pero si esta relacionada con la cantidad de pagos: con 0 o 1 pago, el sobrepago tiene una media de 0; con 2 pagos sube a 0.72, con 3 o mas pagos, practicamente es 1 osea 100% de las facturas quedan sobrepagadas. El problema no parece ser un monto mal calculado, sino que cada pago adicional casi nunca se ajusta al saldo restante.

**Decision (refinada):** Ajustar los montos en `payments` para que la suma por factura no supere el `total`. La relacion con multiples pagos (>=2) explica el sobrepago: cada pago adicional no se ajustaba al saldo restante

## 7. status=paid con pago < total

In [24]:
underpaid_paid = pay_recon[
    (pay_recon["status"] == "paid") & (pay_recon["paid_sum"] < pay_recon["total_money"])
]

print("paid pero pago < total:", len(underpaid_paid), f"({100*len(underpaid_paid)/len(pay_recon):.1f}%)")
print("\nSolapamiento con sobrepagadas:", len(set(overpaid["invoice_id"]) & set(underpaid_paid["invoice_id"])))

paid pero pago < total: 14481 (29.0%)

Solapamiento con sobrepagadas: 0


14481 invoice marcadas `paid` con pago acumulado menor al total.

**Decision :** Corregir `status` cuando una factura esta marcada `paid` pero el pago acumulado es menor al `total` .

### 7.1 Evidencia adicional: ¿el problema crece con el tiempo?

Igual que en las situaciones anteriores, revisamos si los casos `status=paid` con pago menor al total se concentran o aumentan en algun periodo reciente.

In [25]:
pay_recon2["underpaid_paid"] = (pay_recon2["status"] == "paid") & (pay_recon2["paid_sum"] < pay_recon2["total_money"])

print("Tasa de status=paid con pago < total, por anio de issued_at:")
print(pay_recon2.groupby("issued_year")["underpaid_paid"].mean())

Tasa de status=paid con pago < total, por anio de issued_at:
issued_year
2022    0.289509
2023    0.286972
2024    0.289232
2025    0.292777
Name: underpaid_paid, dtype: float64


La tasa se mantiene estable entre 28.7% y 29.3% en los 4 años, sin tendencia. Al igual que en las situaciones 4 y 5, es una caracteristica constante del origen, no un problema que este empeorando.

**Decision (confirmada):** Corregir `status` en las 14 481 facturas afectadas. La tasa estable (~29%) confirma que es caracteristica del origen.

## 8. Suma de weights por enrollment != 1

In [27]:
grades = load_raw("university", "grades")

weight_sum = grades.groupby("enrollment_id")["weight"].apply(
    lambda s: sum(to_decimal(x) for x in s)
)
n_grades = grades.groupby("enrollment_id").size()

summary = pd.DataFrame({"weight_sum": weight_sum, "n_grades": n_grades})
bad_weights = summary[summary["weight_sum"] != Decimal("1")]
good_weights = summary[summary["weight_sum"] == Decimal("1")]

print("Enrollments con suma != 1:", len(bad_weights), "/", len(summary))
print("Promedio de grades (mal):", round(bad_weights["n_grades"].mean(), 2))
print("Promedio de grades (ok):", round(good_weights["n_grades"].mean(), 2))


Enrollments con suma != 1: 22646 / 22786
Promedio de grades (mal): 2.63
Promedio de grades (ok): 3.34


99% de enrollments no suman 1. Los casos tipicos tienen 1-2 notas y suma 0,43-0,49 (evaluacion **incompleta**, no pesos mal normalizados). Solo 140 enrollments suman exactamente 1.

**Decision :** Renormalizar los `weight` por `enrollment_id` para que sumen 1, solamente aquellos enrollment con status completed.

### 8.1 Evidencia adicional: ¿son semestres en curso con notas pendientes?

Una explicacion posible es que los semestres mas recientes aun no tengan todas las notas cargadas (el curso sigue en curso). Si fuera asi, esperariamos que los semestres antiguos (ya terminados) tuvieran `weight_sum = 1` con mucha mas frecuencia que los recientes.

In [16]:
enrollments = load_raw("university", "enrollments")
semesters = load_raw("university", "semesters")

resumen_semestre = summary.reset_index().merge(
    enrollments[["enrollment_id", "semester_id"]], on="enrollment_id", how="left"
).merge(semesters[["semester_id", "year", "half"]], on="semester_id", how="left")

resumen_semestre["completo"] = resumen_semestre["weight_sum"] == Decimal("1")

print("Tasa de enrollments con weight_sum == 1, por semestre (year, half):")
print(resumen_semestre.groupby(["year", "half"])["completo"].mean())

Tasa de enrollments con weight_sum == 1, por semestre (year, half):
year  half
2022  1      0.01
      2      0.01
2023  1      0.00
      2      0.01
2024  1      0.01
      2      0.00
2025  1      0.01
      2      0.01
Name: completo, dtype: float64


La tasa de `weight_sum == 1` es baja (0.4%-0.8%) en todos los semestres, incluidos los mas antiguos ya terminados (2022-1). Si el problema fuera solo evaluaciones en curso, los semestres viejos deberian estar casi completos, y no es el caso. Se descarta la hipotesis de "semestre en curso": la incompletitud parece ser una caracteristica estructural del dataset. 

**Decision (confirmada):** Renormalizar los `weight` por enrollment para que sumen 1. La baja tasa de completitud en todos los semestres (incluidos los cerrados) descarta la hipotesis de curso en progreso.